In [1]:
from dotenv import load_dotenv

# Path to env file containing the waterbodies database credentials
# Only necessary on the Sandbox.
dotenv_path = "/home/jovyan/.env"
load_dotenv(dotenv_path=dotenv_path, verbose=True, override=True)

True

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np

In [3]:
vector_path = "map.geojson"
# Filter waterbodies by area (m2).
max_area = np.inf
min_area = 5000
# Timeseries
start_date = "2021-10-01"
end_date = "2022-01-31"

In [4]:
# Load the area of interest polygon.
aoi_gdf = gpd.read_file(vector_path).to_crs("EPSG:4326")

In [5]:
def get_waterbodies_api(aoi_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    # Get the bounding box of the polygon
    minx, miny, maxx, maxy = aoi_gdf.total_bounds
    
    # Make an API request to get all waterbody polygons in the bounding box of the polygon and store as a geodataframe.
    from waterbodies.db import get_waterbodies_engine
    engine = get_waterbodies_engine()
    sql = f"SELECT * from waterbodies_historical_extent WHERE geometry && ST_MakeEnvelope({minx}, {miny}, {maxx}, {maxy}, 4326);"
    waterbodies = gpd.read_postgis(sql=sql, con=engine, geom_col="geometry")
    
    # Filter the waterbody polygons to those located within the supplied polygon.
    # TODO: Should we use within instead?
    intersecting_waterbodies_uids = waterbodies.sjoin(aoi_gdf, how="inner", predicate="intersects")["uid"].unique()
    waterbodies = waterbodies[waterbodies["uid"].isin(intersecting_waterbodies_uids)].reset_index(drop=True)
        
    return waterbodies

In [6]:
waterbodies = get_waterbodies_api(aoi_gdf=aoi_gdf)
waterbodies

,uid,wb_id,area_m2,length_m,perim_m,geometry
0,edu6c02wpf,49169,6.300000e+03,120.000000,3.600000e+02,"POLYGON ((-16.47903 15.95479, -16.47903 15.954..."
1,edu62ebzqt,49121,2.430000e+04,300.000000,8.400000e+02,"POLYGON ((-16.49986 15.88737, -16.49955 15.887..."
2,edu62kng16,49122,8.100000e+03,270.000000,6.000000e+02,"POLYGON ((-16.50328 15.88786, -16.50328 15.885..."
3,edu62te906,49123,8.460000e+04,698.030525,1.800000e+03,"POLYGON ((-16.49675 15.89714, -16.49582 15.897..."
4,edu689r7kw,49147,4.320000e+04,904.215431,2.340000e+03,"POLYGON ((-16.49146 15.91937, -16.49115 15.919..."
...,...,...,...,...,...,...
130,edu7d23f8w,49266,9.000000e+03,120.000000,4.800000e+02,"POLYGON ((-16.42244 16.08629, -16.42182 16.086..."
131,edu7d2t7ep,49267,1.737000e+05,1026.061616,3.840000e+03,"POLYGON ((-16.41560 16.09020, -16.41529 16.090..."
132,edu7d3mu35,49268,8.010000e+04,360.033782,1.440000e+03,"POLYGON ((-16.41622 16.09289, -16.41529 16.092..."
133,edu7d8bnxy,49271,9.900000e+03,119.999878,4.200000e+02,"POLYGON ((-16.41373 16.08971, -16.41373 16.088..."


In [7]:
# Filter waterbodies by size
waterbodies = waterbodies[(waterbodies['area_m2'] >= min_area) & (waterbodies['area_m2'] <= max_area)].sort_values("area_m2").reset_index(drop=True)
waterbodies

,uid,wb_id,area_m2,length_m,perim_m,geometry
0,edu63t795z,49138,5.400000e+03,150.000000,4.200000e+02,"POLYGON ((-16.45260 15.89396, -16.45229 15.893..."
1,edu689933h,49145,5.400000e+03,180.000000,4.200000e+02,"POLYGON ((-16.49986 15.91741, -16.49986 15.915..."
2,edu74u5xey,49222,5.400000e+03,90.000000,3.600000e+02,"POLYGON ((-16.39756 16.01979, -16.39725 16.019..."
3,edu6fgvrse,49193,5.400000e+03,147.580596,4.200000e+02,"POLYGON ((-16.39508 15.97458, -16.39477 15.974..."
4,edu63r66e0,49136,6.300000e+03,169.705627,4.800000e+02,"POLYGON ((-16.46597 15.90495, -16.46597 15.904..."
...,...,...,...,...,...,...
130,edu6937xr5,49160,2.407500e+06,3679.252504,1.812000e+04,"POLYGON ((-16.46224 15.93036, -16.46193 15.930..."
131,edu76r1p9v,49244,2.489400e+06,6041.129510,2.754000e+04,"POLYGON ((-16.39974 16.09435, -16.39912 16.094..."
132,edu68ywn2m,49155,2.717100e+06,5223.554402,3.996000e+04,"POLYGON ((-16.48556 15.96261, -16.48493 15.962..."
133,edu714mv1c,49203,8.721000e+06,7814.872332,3.336000e+04,"POLYGON ((-16.46224 16.03666, -16.46193 16.036..."


In [8]:
# View the waterbodies
m = aoi_gdf.explore(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="ESRI WorldImagery",
    color='black',
    style_kwds={'fillOpacity': 0, 'weight': 2},)
waterbodies.explore(
    m=m,
    color='blue',
    style_kwds={'fillOpacity': 0.3, 'weight': 1},)

In [9]:
def get_timeseries_api(wb_id: int, start_date: str, end_date: str) -> pd.DataFrame:
    part1 = f"WITH wb AS (SELECT uid, area_m2 as actual_area_m2 from waterbodies_historical_extent where wb_id = {wb_id})"
    part2 = f"wbo AS (SELECT wo.*, wb.actual_area_m2 FROM waterbodies_observations AS wo INNER JOIN wb ON wo.uid = wb.uid WHERE wo.date BETWEEN '{start_date}' AND '{end_date}')"
    part3 = f"waterbody_stats AS (SELECT date, SUM(area_wet_m2) AS area_wet_m2, SUM(area_dry_m2) AS area_dry_m2, SUM(area_invalid_m2) AS area_invalid_m2, SUM(area_wet_m2 + area_dry_m2 + area_invalid_m2) AS area_observed_m2, actual_area_m2 FROM wbo GROUP BY date, actual_area_m2)"
    part4 = f"waterbody_stats_pc AS (SELECT date, area_wet_m2, (area_wet_m2/actual_area_m2) * 100 AS percent_wet, area_dry_m2, (area_dry_m2/actual_area_m2) * 100 AS percent_dry, area_invalid_m2, (area_invalid_m2/actual_area_m2) * 100 AS percent_invalid, area_observed_m2, (area_observed_m2/actual_area_m2) * 100 AS percent_observed FROM waterbody_stats)"
    part5 = f"filtered_stats AS (SELECT * FROM waterbody_stats_pc WHERE percent_observed > 85 AND percent_invalid < 100)"
    sql = f"{part1}, {part2}, {part3}, {part4}, {part5} SELECT * from filtered_stats ORDER BY date"
    
    from waterbodies.db import get_waterbodies_engine
    engine = get_waterbodies_engine()
    
    waterbodies_timeseries = pd.read_sql(sql=sql, con=engine)
    return waterbodies_timeseries

In [10]:
%%time
# Request the time series of a user supplied time range (suggest three months in a particular season (either wet or dry)
# Calculate the average or median wet surface percentage for the time frame and add it as an attribute to the geodataframe
waterbodies["avg_pc_wet"] = waterbodies["wb_id"].apply(lambda x: get_timeseries_api(wb_id=x, start_date=start_date, end_date=end_date)["percent_wet"].mean())

CPU times: user 912 ms, sys: 104 ms, total: 1.02 s
Wall time: 29.4 s


In [11]:
# Display a table with the summary statistic for each water body
waterbodies

,uid,wb_id,area_m2,length_m,perim_m,geometry,avg_pc_wet
0,edu63t795z,49138,5.400000e+03,150.000000,4.200000e+02,"POLYGON ((-16.45260 15.89396, -16.45229 15.893...",47.222222
1,edu689933h,49145,5.400000e+03,180.000000,4.200000e+02,"POLYGON ((-16.49986 15.91741, -16.49986 15.915...",0.000000
2,edu74u5xey,49222,5.400000e+03,90.000000,3.600000e+02,"POLYGON ((-16.39756 16.01979, -16.39725 16.019...",0.000000
3,edu6fgvrse,49193,5.400000e+03,147.580596,4.200000e+02,"POLYGON ((-16.39508 15.97458, -16.39477 15.974...",31.481481
4,edu63r66e0,49136,6.300000e+03,169.705627,4.800000e+02,"POLYGON ((-16.46597 15.90495, -16.46597 15.904...",7.142857
...,...,...,...,...,...,...,...
130,edu6937xr5,49160,2.407500e+06,3679.252504,1.812000e+04,"POLYGON ((-16.46224 15.93036, -16.46193 15.930...",64.845794
131,edu76r1p9v,49244,2.489400e+06,6041.129510,2.754000e+04,"POLYGON ((-16.39974 16.09435, -16.39912 16.094...",29.440829
132,edu68ywn2m,49155,2.717100e+06,5223.554402,3.996000e+04,"POLYGON ((-16.48556 15.96261, -16.48493 15.962...",18.660604
133,edu714mv1c,49203,8.721000e+06,7814.872332,3.336000e+04,"POLYGON ((-16.46224 16.03666, -16.46193 16.036...",46.830925


In [15]:
# Make an interactive map that displays the water body polygons, colour coded by wet surface area summary statistic that was just calculated
waterbodies.explore(
    column="avg_pc_wet",
    cmap="viridis",
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="ESRI WorldImagery",)